In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer


In [ ]:
ds = Dataset.from_parquet('../dataset/finetune/train-00000-of-00001-bb5f874d67d84fd2.parquet')

# 模型创建

In [2]:
tokenizer = AutoTokenizer.from_pretrained("../models/Qwen/Qwen3-4B")
model = AutoModelForCausalLM.from_pretrained("../models/Qwen/Qwen3-4B", device_map='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

# 规整函数，处理成支持的输入

In [ ]:
ds[0]

In [ ]:
for name, _ in model.named_parameters():
    print(name)

In [ ]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["INSTRUCTION"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["RESPONSE"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [ ]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

# 解码效果测试(token_id 2 text)

In [ ]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

In [ ]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM, target_modules=".*self_attn\.\w_proj", r=6)
config

In [ ]:
model = get_peft_model(model, config)

In [ ]:
config

In [ ]:
model

In [ ]:
model.print_trainable_parameters()

In [ ]:
args = TrainingArguments(
    output_dir="./cache",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1
)

In [ ]:
train_test_dict = tokenized_ds.train_test_split(train_size=0.8)
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=train_test_dict['train'],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

In [ ]:
trainer.train()

# 推理

In [3]:
from peft import PeftModel

In [4]:
p_model = PeftModel.from_pretrained(model, model_id="./cache/checkpoint-1618", device_map='auto')

ipt = tokenizer("Human: {}\n{}".format('how can i make up a linux service quickly?', "").strip() + "\n\nAssistant: ", return_tensors="pt")
tokenizer.decode(p_model.generate(**ipt, do_sample=False)[0], skip_special_tokens=True)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
D:\Data\CodeRepository\py\zhy-dl\.venv\Lib\site-packages\transformers\generation\utils.py:2532: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


'Human: how can i make up a linux service quickly?\n\nAssistant: you can use the service command'

# 模型合并与保存

In [5]:
merge_model = p_model.merge_and_unload()
merge_model.save_pretrained("./chatbot/merge_model")